# CausalPype Demo: Wine Quality 

**Dataset**: UCI Wine Quality — Red Wine

**Causal questions**:
- How does alcohol content causally affect wine quality ratings?
- Which chemical properties have the strongest causal influence on quality?
- What is the dose-response curve for alcohol → quality?
- Can we detect anomalous wines and attribute the anomaly to specific chemical drivers?

**Tasks demonstrated**: ATE, Dose-Response, Arrow Strength, Intrinsic Causal Influence, Intervention, Stochastic Intervention, Anomaly Attribution, Counterfactual, Validate

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Download UCI Wine Quality (Red)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine = pd.read_csv(url, sep=";")
print(f"Loaded {len(wine)} red wines, {len(wine.columns)} features")
wine.head()

Loaded 1599 red wines, 12 features


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## Data Preparation & Causal Graph

We define a causal DAG grounded in wine chemistry:
- **fixed_acidity** → volatile_acidity, density, pH, quality
- **volatile_acidity** → quality (vinegar taste = bad)
- **citric_acid** → pH, volatile_acidity (antimicrobial)
- **residual_sugar** → density, alcohol (fermentation consumes sugar to produce alcohol)
- **alcohol** → quality (strong known predictor)
- **sulphates** → quality (antimicrobial preservative)
- **pH** → quality
- **density** → alcohol

In [2]:
import causalpype as cp

# Select features with clear causal relationships and rename for clean column names
df = wine[["fixed acidity", "volatile acidity", "citric acid", "residual sugar",
           "sulphates", "pH", "density", "alcohol", "quality"]].copy()
df.columns = ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
              "sulphates", "pH", "density", "alcohol", "quality"]

# Causal DAG based on wine chemistry domain knowledge
graph = {
    "fixed_acidity": ["volatile_acidity", "density", "pH", "quality"],
    "citric_acid": ["pH", "volatile_acidity"],
    "volatile_acidity": ["quality"],
    "residual_sugar": ["density", "alcohol"],
    "density": ["alcohol"],
    "alcohol": ["quality"],
    "sulphates": ["quality"],
    "pH": ["quality"],
}

model = cp.CausalModel(graph)


In [3]:
model.fit(df)
print(model)

Fitting causal mechanism of node quality: 100%|██████████| 9/9 [00:08<00:00,  1.06it/s]         

CausalModel(fitted, nodes=['fixed_acidity', 'citric_acid', 'volatile_acidity', 'residual_sugar', 'density', 'alcohol', 'sulphates', 'pH', 'quality'], edges=[('fixed_acidity', 'volatile_acidity'), ('fixed_acidity', 'density'), ('fixed_acidity', 'pH'), ('fixed_acidity', 'quality'), ('citric_acid', 'pH'), ('citric_acid', 'volatile_acidity'), ('volatile_acidity', 'quality'), ('residual_sugar', 'density'), ('residual_sugar', 'alcohol'), ('density', 'alcohol'), ('alcohol', 'quality'), ('sulphates', 'quality'), ('pH', 'quality')])


## Task 1: ATE — Does Alcohol Increase Quality?

Comparing do(alcohol=11) vs do(alcohol=9) — a moderate increase in alcohol content.

In [3]:
ate_result = model.run(cp.ATE(
    treatment="alcohol",
    outcome="quality",
    treatment_value=11.0,
    control_value=9.0,
    num_samples=3000,
))
print(ate_result)

╭──────────────────── ATE ────────────────────╮
│                                             │
│  Estimate: ↑ 0.4033                         │
│   Treatment                     alcohol     │
│   Outcome                       quality     │
│   Treatment Value               ↑ 11.0000   │
│   Control Value                 ↑ 9.0000    │
│   Num Samples                   3,000       │
│                                             │
╰─────────────────────────────────────────────╯

In [4]:
ate_result = model.run(cp.CATE(
    treatment="alcohol",
    outcome="quality",
    treatment_value=11.0,
    control_value=9.0,
    num_samples=3000,
))
print(ate_result)

TypeError: CATE.__init__() missing 1 required positional argument: 'effect_modifiers'

## Task 2: Dose-Response Curve

How does expected wine quality change as we sweep alcohol from 8% to 14%?

In [4]:
dose_result = model.run(cp.DoseResponse(
    treatment="alcohol",
    outcome="quality",
    treatment_values=np.linspace(8, 14, 15),
    num_samples=1000,
))

# The estimate is a DataFrame with the dose-response curve
print(dose_result.estimate.to_string(index=False))

 treatment_value  expected_outcome      std
        8.000000             5.206 0.577550
        8.428571             5.189 0.548889
        8.857143             5.208 0.602276
        9.285714             5.205 0.582216
        9.714286             5.288 0.589115
       10.142857             5.560 0.674092
       10.571429             5.849 0.556955
       11.000000             5.592 0.702521
       11.428571             5.840 0.666633
       11.857143             6.172 0.595328
       12.285714             6.318 0.637868
       12.714286             6.454 0.638658
       13.142857             6.238 0.676281
       13.571429             6.322 0.659027
       14.000000             6.317 0.668215


## Task 3: Arrow Strength

Which incoming edges have the strongest causal effect on quality?

In [5]:
arrow_result = model.run(cp.ArrowStrength(target="quality"))
print(arrow_result)

╭───────────────────────── Arrow Strength ──────────────────────────╮
│                                                                   │
│  Estimate:                                                        │
│   Target                        quality                           │
│   alcohol -> quality             ↑ 0.2481  ████████████████████   │
│   sulphates -> quality           ↑ 0.1740  ██████████████         │
│   volatile_acidity -> quality    ↑ 0.1159  █████████              │
│   pH -> quality                  ↑ 0.0927  ███████                │
│   fixed_acidity -> quality       ↑ 0.0773  ██████                 │
│                                                                   │
╰───────────────────────────────────────────────────────────────────╯

## Task 4: Intrinsic Causal Influence

Decompose the variance in quality — how much comes from each upstream noise term?

In [6]:
ici_result = model.run(cp.IntrinsicCausalInfluence(target="quality"))
print(ici_result)

Evaluating set functions...: 100%|██████████| 150/150 [00:47<00:00,  3.14it/s]


╭──────────────── Intrinsic Causal Influence ────────────────╮
│                                                            │
│  Estimate:                                                 │
│   Target                        quality                    │
│   Total Variance Explained      ↑ 0.3356                   │
│   quality                 ↑ 0.1546  ████████████████████   │
│   sulphates               ↑ 0.0578  ███████                │
│   alcohol                 ↑ 0.0394  █████                  │
│   fixed_acidity           ↑ 0.0297  ███                    │
│   density                 ↑ 0.0262  ███                    │
│   residual_sugar          ↑ 0.0120  █                      │
│   citric_acid             ↑ 0.0086  █                      │
│   volatile_acidity        ↑ 0.0064                         │
│   pH                      ↑ 0.0010                         │
│                                                            │
│  Normalized                                                │
│   quality                    46.1%  ████████████████████   │
│   sulphates                  17.2%  ███████                │
│   alcohol                    11.7%  █████                  │
│   fixed_acidity               8.8%  ███                    │
│   density                     7.8%  ███                    │
│   residual_sugar              3.6%  █                      │
│   citric_acid                 2.6%  █                      │
│   volatile_acidity            1.9%                         │
│   pH                          0.3%                         │
│                                                            │
╰────────────────────────────────────────────────────────────╯

## Task 5: Interventional Samples

What quality distribution would we get if we set alcohol=12 and volatile_acidity=0.3?

In [7]:
# Intervention: set alcohol=12 AND volatile_acidity=0.3 (low = good)
interv_result = model.run(cp.Intervention(
    interventions={"alcohol": 12.0, "volatile_acidity": 0.3},
    outcome="quality",
    num_samples=3000,
))
print(interv_result)

╭─────────────────────── Intervention ───────────────────────╮
│                                                            │
│  Estimate: ↑ 6.3437                                        │
│   Outcome                       quality                    │
│   Mean                          ↑ 6.3437                   │
│   STD                           ↑ 0.6173                   │
│                                                            │
│  Interventions                                             │
│   alcohol                ↑ 12.0000  ████████████████████   │
│   volatile_acidity        ↑ 0.3000                         │
│                                                            │
╰────────────────────────────────────────────────────────────╯

## Task 6: Stochastic Intervention

Instead of hard-setting alcohol, what if we nudge it up slightly (+0.5 units)?

In [8]:
stoch_result = model.run(cp.StochasticIntervention(
    treatment="alcohol",
    outcome="quality",
    shift=0.5,  # add 0.5% alcohol to natural value
    num_samples=3000,
))
print(stoch_result)

╭───────── Stochastic Intervention ──────────╮
│                                            │
│  Estimate: ↑ 0.1617                        │
│   Treatment                     alcohol    │
│   Outcome                       quality    │
│   Shift                         ↑ 0.5000   │
│   Is Binary                     ✗          │
│   E[Y|Baseline]                 ↑ 5.6293   │
│   E[Y|Shifted]                  ↑ 5.7910   │
│   Effect                        ↑ 0.1617   │
│                                            │
╰────────────────────────────────────────────╯

## Task 7: Anomaly Attribution

Which chemical properties are responsible for anomalously-rated wines?

In [ ]:
anomaly_result = model.run(cp.AnomalyAttribution(
    target="quality",
    anomaly_threshold_percentile=95,
))
print(anomaly_result)

Evaluating set functions...:  75%|███████▌  | 112/149 [08:50<03:07,  5.07s/it]

## Task 8: Counterfactual — What if This Specific Wine Had Less Volatile Acidity?

Pick a low-quality wine and ask: what would its quality have been with lower volatile acidity?

In [ ]:
# Pick 5 low-quality wines (quality <= 4)
low_quality_wines = df[df["quality"] <= 4].head(5)
print("Selected low-quality wines:")
print(low_quality_wines[["volatile_acidity", "alcohol", "quality"]].to_string())

# Counterfactual: what if these wines had volatile_acidity = 0.3 (a good level)?
cf_result = model.run(cp.Counterfactual(
    interventions={"volatile_acidity": 0.3},
    observed_data=low_quality_wines,
    outcome="quality",
))
print("\n")
print(cf_result)

Selected low-quality wines:
    volatile_acidity  alcohol  quality
18             0.590      9.0        4
38             1.130      9.8        4
41             0.610      9.3        4
45             0.520     13.1        4
73             0.675      9.2        4


Counterfactual Analysis
───────────────────────
  Interventions:               volatile_acidity=0.3
  N units:                     5
  Outcome:                     quality

  Factual Mean:                4.00
  Counterfactual Mean:         4.40
  Average Effect:              ↑ 0.40


## Task 9: Distribution Change

We split the data into two eras and ask: what caused the shift in quality distribution?

In [ ]:
# Split data into two halves (simulating two production periods)
old_data = df.iloc[:800]
new_data = df.iloc[800:]

print(f"Old batch mean quality: {old_data['quality'].mean():.2f}")
print(f"New batch mean quality: {new_data['quality'].mean():.2f}")

dist_result = model.run(cp.DistributionChange(
    target="quality",
    old_data=old_data,
    new_data=new_data,
))
print("\n")
print(dist_result)

Old batch mean quality: 5.54
New batch mean quality: 5.73


Evaluating set functions...: 100%|██████████| 151/151 [00:10<00:00, 14.23it/s]




Distribution Change → quality
─────────────────────────────
  Old: n=800  |  New: n=799

  alcohol                                  0.0000  
  citric_acid                              0.0000  
  density                                  0.0000  
  fixed_acidity                            0.0000  
  pH                                       0.0000  
  quality                                  0.0000  
  residual_sugar                           0.0000  
  sulphates                                0.0000  
  volatile_acidity                         0.0000  


## Task 10: Full Pipeline + Validate

Run validation checks and a multi-task pipeline in one call.

In [ ]:
# Run a full pipeline combining multiple analyses
results = model.run([
    cp.ATE("alcohol", "quality", treatment_value=12.0, control_value=9.0),
    cp.ATE("volatile_acidity", "quality", treatment_value=0.8, control_value=0.3),
    cp.ArrowStrength("quality"),
    cp.Intervention({"alcohol": 13.0, "sulphates": 0.8}, outcome="quality"),
])

# Print the full unified report
print(model.report())

CausalPype Analysis Report

--- Causal Graph ---
  Nodes (9): ['fixed_acidity', 'citric_acid', 'volatile_acidity', 'residual_sugar', 'density', 'alcohol', 'sulphates', 'pH', 'quality']
  Edges (13): [('fixed_acidity', 'volatile_acidity'), ('fixed_acidity', 'density'), ('fixed_acidity', 'pH'), ('fixed_acidity', 'quality'), ('citric_acid', 'pH'), ('citric_acid', 'volatile_acidity'), ('volatile_acidity', 'quality'), ('residual_sugar', 'density'), ('residual_sugar', 'alcohol'), ('density', 'alcohol'), ('alcohol', 'quality'), ('sulphates', 'quality'), ('pH', 'quality')]
  Root nodes: ['fixed_acidity', 'citric_acid', 'residual_sugar', 'sulphates']
  Leaf nodes: ['quality']

--- Data ---
  Rows: 1599
  Columns: 9

ATE: alcohol → quality
──────────────────────
  Contrast:                    9.0 → 12.0
  Estimate:                    ↑ 0.91
  Samples:                     2000

ATE: volatile_acidity → quality
───────────────────────────────
  Contrast:                    0.3 → 0.8
  Estimate:    

In [ ]:
# Validate model assumptions
validation = model.validate()
print(validation)

Model Validation: ISSUES FOUND
──────────────────────────────

  Edge tests: 12 passed, 1 failed
  ✓ alcohol -> quality                       (p=0.0000)
  ✓ citric_acid -> pH                        (p=0.0000)
  ✓ citric_acid -> volatile_acidity          (p=0.0000)
  ✓ density -> alcohol                       (p=0.0000)
  ✓ fixed_acidity -> density                 (p=0.0000)
  ✓ fixed_acidity -> pH                      (p=0.0000)
  ✓ fixed_acidity -> quality                 (p=0.0000)
  ✓ fixed_acidity -> volatile_acidity        (p=0.0000)
  ✗ pH -> quality                            (p=0.3517)
  ✓ residual_sugar -> alcohol                (p=0.0000)
  ✓ residual_sugar -> density                (p=0.0000)
  ✓ sulphates -> quality                     (p=0.0000)
  ✓ volatile_acidity -> quality              (p=0.0000)

  Local Markov tests:
  ✗ volatile_acidity                         (p=0.0000)
  ✗ density                                  (p=0.0000)
  ✗ alcohol                             